# 06 — A0 vs A1 Controlled Comparison

This notebook performs the first formal ablation comparison:

**A1 − A0 = effect of explicit phase-aware enhancement**

It expects:

- `D:\PAPERS\SPEECH\low_snr_speech_enhancement\outputs\a0_voicebank\evaluation\per_utterance.csv`
- `D:\PAPERS\SPEECH\low_snr_speech_enhancement\outputs\a1_phaseaware\evaluation\per_utterance.csv`

The same utterance IDs must be present in both files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

PROJECT_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement")

A0_FILE = (
    PROJECT_ROOT /
    "outputs" /
    "a0_voicebank" /
    "evaluation" /
    "per_utterance.csv"
)

A1_FILE = (
    PROJECT_ROOT /
    "outputs" /
    "a1_phaseaware" /
    "evaluation" /
    "per_utterance.csv"
)

assert A0_FILE.exists(), f"Missing: {A0_FILE}"
assert A1_FILE.exists(), f"Missing: {A1_FILE}"

a0 = pd.read_csv(A0_FILE)
a1 = pd.read_csv(A1_FILE)

print("A0 utterances:", len(a0))
print("A1 utterances:", len(a1))

## Merge by utterance ID and verify paired evaluation

In [ ]:
metrics = [
    "pesq",
    "stoi",
    "estoi",
    "si_sdr",
    "snr_ref"
]

keep0 = [
    "utt_id"
] + [
    f"enh_{m}"
    for m in metrics
]

keep1 = [
    "utt_id"
] + [
    f"enh_{m}"
    for m in metrics
]

paired = a0[keep0].merge(
    a1[keep1],
    on="utt_id",
    how="inner",
    suffixes=("_a0", "_a1")
)

assert len(paired) == len(a0) == len(a1), (
    "A0 and A1 do not contain the same frozen test utterances."
)

print("Paired utterances:", len(paired))
display(paired.head())

## Paired bootstrap confidence intervals and Wilcoxon tests

The bootstrap operates on per-utterance A1−A0 differences.

Holm correction is applied across the five metric comparisons.

In [ ]:
def paired_bootstrap_ci(
    differences,
    n_boot=10000,
    seed=2026,
    alpha=0.05
):
    x = np.asarray(
        differences,
        dtype=np.float64
    )

    x = x[np.isfinite(x)]

    rng = np.random.default_rng(
        seed
    )

    means = np.empty(
        n_boot,
        dtype=np.float64
    )

    n = len(x)

    for i in range(n_boot):
        idx = rng.integers(
            0,
            n,
            size=n
        )
        means[i] = x[idx].mean()

    lo = np.quantile(
        means,
        alpha / 2
    )

    hi = np.quantile(
        means,
        1 - alpha / 2
    )

    return (
        float(x.mean()),
        float(lo),
        float(hi)
    )

def holm_adjust(pvalues):
    pvalues = np.asarray(
        pvalues,
        dtype=float
    )

    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty(m)

    running_max = 0.0

    for rank, idx in enumerate(order):
        value = (
            (m - rank) *
            pvalues[idx]
        )
        running_max = max(
            running_max,
            value
        )
        adjusted[idx] = min(
            running_max,
            1.0
        )

    return adjusted

rows = []
raw_p = []

for metric in metrics:
    a0_values = pd.to_numeric(
        paired[f"enh_{metric}_a0"],
        errors="coerce"
    ).to_numpy()

    a1_values = pd.to_numeric(
        paired[f"enh_{metric}_a1"],
        errors="coerce"
    ).to_numpy()

    valid = (
        np.isfinite(a0_values) &
        np.isfinite(a1_values)
    )

    a0_values = a0_values[valid]
    a1_values = a1_values[valid]

    diff = (
        a1_values -
        a0_values
    )

    mean_diff, ci_lo, ci_hi = (
        paired_bootstrap_ci(diff)
    )

    # Rank-biserial style paired effect estimate:
    # proportion positive minus proportion negative.
    positive = np.sum(diff > 0)
    negative = np.sum(diff < 0)

    effect = (
        (positive - negative) /
        max(positive + negative, 1)
    )

    try:
        test = wilcoxon(
            a1_values,
            a0_values,
            zero_method="wilcox",
            alternative="two-sided"
        )
        p = float(test.pvalue)
    except ValueError:
        p = np.nan

    raw_p.append(p)

    rows.append({
        "metric": metric,
        "a0_mean": float(np.mean(a0_values)),
        "a1_mean": float(np.mean(a1_values)),
        "mean_difference_a1_minus_a0": mean_diff,
        "bootstrap_95ci_low": ci_lo,
        "bootstrap_95ci_high": ci_hi,
        "pct_utterances_a1_better": float(
            100.0 *
            np.mean(diff > 0)
        ),
        "paired_sign_effect": effect,
        "wilcoxon_p_raw": p,
        "n": len(diff)
    })

comparison = pd.DataFrame(rows)

finite_p = np.array([
    p if np.isfinite(p) else 1.0
    for p in raw_p
])

comparison["wilcoxon_p_holm"] = (
    holm_adjust(finite_p)
)

display(comparison)

OUTPUT_FILE = (
    PROJECT_ROOT /
    "outputs" /
    "a1_phaseaware" /
    "a0_vs_a1_comparison.csv"
)

comparison.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)

## Automatic interpretation guardrail

In [ ]:
for _, row in comparison.iterrows():
    metric = row["metric"]
    diff = row["mean_difference_a1_minus_a0"]
    lo = row["bootstrap_95ci_low"]
    hi = row["bootstrap_95ci_high"]
    p_holm = row["wilcoxon_p_holm"]

    if lo > 0 and p_holm < 0.05:
        verdict = "A1 shows a consistent improvement over A0."
    elif hi < 0 and p_holm < 0.05:
        verdict = "A1 shows a consistent degradation relative to A0."
    else:
        verdict = "No clear paired improvement is established."

    print(
        f"{metric:8s} | "
        f"Δ={diff:+.5f} | "
        f"95% CI [{lo:+.5f}, {hi:+.5f}] | "
        f"Holm p={p_holm:.3e} | "
        f"{verdict}"
    )